# pandas: detailed scenarios

A runnable tour of pandas on a small synthetic price panel. Self-contained. See **pandas_cheatsheet.md** for the full reference.

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(1)

dates = pd.bdate_range("2023-01-02", periods=180)
def walk(p0, mu, sig):
    return p0 * np.cumprod(1 + rng.normal(mu, sig, len(dates)))

prices = pd.DataFrame({
    "ACME": walk(98, 0.0004, 0.014),
    "GLOBEX": walk(51, 0.0002, 0.010),
    "INITECH": walk(210, 0.0009, 0.020),
}, index=dates)
prices.index.name = "date"
prices.iloc[20, prices.columns.get_loc("INITECH")] = np.nan  # a deliberate gap
prices.head()

## Inspect

In [ ]:
print(prices.shape)
print(prices.dtypes.to_dict())
prices.describe().round(2)

## Clean: fill the gap forward

In [ ]:
print("NaNs before:", int(prices.isna().sum().sum()))
prices = prices.sort_index().ffill()
print("NaNs after:", int(prices.isna().sum().sum()))

## Select and filter

In [ ]:
june_acme = prices.loc["2023-06", "ACME"]
print("June ACME rows:", len(june_acme))
big_moves = prices["ACME"].pct_change().abs() > 0.02
print("days ACME moved more than 2 percent:", int(big_moves.sum()))

## Derive returns

In [ ]:
returns = prices.pct_change().dropna()
returns.head(3).round(4)

## Time series: resample, rolling, ewm

In [ ]:
monthly = prices.resample("M").last()
sma50 = prices["ACME"].rolling(50).mean()
ema20 = prices["ACME"].ewm(span=20).mean()
print("monthly rows:", len(monthly))
print("last SMA50:", round(float(sma50.iloc[-1]), 2), " last EMA20:", round(float(ema20.iloc[-1]), 2))

## Group and aggregate (long format)

Melt to long, then group by ticker.

In [ ]:
long = returns.reset_index().melt(id_vars="date", var_name="ticker", value_name="ret")
summary = long.groupby("ticker")["ret"].agg(["mean", "std", "count"]).sort_values("std")
summary.round(4)

## Reshape: long back to wide

In [ ]:
wide = long.pivot(index="date", columns="ticker", values="ret")
print("wide shape:", wide.shape)
wide.head(2).round(4)

## Rank and top-N

In [ ]:
print("5 best ACME days:")
print(returns["ACME"].nlargest(5).round(4))

## Correlation and rebased performance

In [ ]:
print("correlation:")
print(returns.corr().round(2))
rebased = prices / prices.iloc[0] * 100
print("final rebased levels:")
print(rebased.iloc[-1].round(1))

## Drawdown

In [ ]:
cum = (1 + returns["ACME"]).cumprod()
drawdown = cum / cum.cummax() - 1
print("ACME max drawdown:", round(float(drawdown.min()), 3))

## Recap

Build, inspect, clean, select, returns, time-series ops, groupby, reshape, rank, correlation, rebasing, drawdown.